# How to reduce LLM tokens and latency & Cost

In [1]:
GROQ_API_KEY = "api key"

# Available free-tier models
MODELS = {
    "fast":  "llama-3.1-8b-instant",
    "smart": "llama-3.3-70b-versatile",
}

In [23]:
from groq import Groq
import time

client = Groq(api_key=GROQ_API_KEY)

def call_llm(prompt, model="llama-3.3-70b-versatile"):
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
        
    )

    latency = time.time() - start
    msg     = response.choices[0].message.content
    usage   = response.usage  

    return {
        "output":             msg,
        "latency":            round(latency, 2),
        "prompt_tokens":     usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens":      usage.total_tokens,
    }

# Quick test
result = call_llm("Say hello")
print(result)

{'output': 'Hello. How can I help you today!', 'latency': 0.89, 'prompt_tokens': 37, 'completion_tokens': 10, 'total_tokens': 47}


## 1 ) Baseline ( unoptimized prompt)

In [ ]:
# unoptimized prompt

prompt = """
Explain in detail what is vector in mathematics,
include examples, real life applications, history,
advantages, disadvantages, and deep explanation.
"""

result = call_llm(prompt)

print("Output:", result["output"])
print("Latency:",          result["latency"], "s")
print("Prompt tokens:",    result["prompt_tokens"])
print("Completion tokens:",result["completion_tokens"])
print("Total tokens:",     result["total_tokens"])

Output: **Introduction to Vectors**

In mathematics, a vector is a mathematical object that has both magnitude (length) and direction. It is a fundamental concept in various fields, including physics, engineering, computer science, and mathematics. Vectors are used to represent quantities that have both size and direction, such as forces, velocities, and accelerations.

**Definition and Properties**
-----------------------------

A vector is defined as an ordered pair of numbers, which can be represented graphically as an arrow in a coordinate system. The length of the arrow represents the magnitude of the vector, while the direction of the arrow represents the direction of the vector. Vectors can be added and scaled (multiplied by a number), and these operations follow certain rules.

Some key properties of vectors include:

* **Magnitude**: The length of a vector, denoted by ||v|| or |v|.
* **Direction**: The direction of a vector, which can be represented by an angle or a unit vecto

## 2) Reduce unnecessary words

In [11]:
prompt_optimized = "Explain vector with examples in 5 bullet points."

result_opt = call_llm(prompt_optimized)

print("Output:", result_opt["output"])
print("Total tokens:", result_opt["total_tokens"])

# Compare
diff = result["total_tokens"] - result_opt["total_tokens"]
print(f"Token reduction: {diff} tokens saved")
print("Latency:",          result["latency"], "s")

Output: A vector is a mathematical object that has both magnitude (length) and direction. Here are five examples of vectors in different contexts:

* **Displacement vector**: Imagine you're moving from one point to another in a room. The displacement vector would represent the shortest path between the two points, with its magnitude being the distance traveled and its direction being the direction of movement. For example, if you move 5 meters east, the displacement vector would have a magnitude of 5 meters and a direction of east.
* **Force vector**: When you push or pull an object, you're applying a force to it. The force vector represents the direction and magnitude of that force. For example, if you push a box with a force of 10 Newtons to the right, the force vector would have a magnitude of 10 Newtons and a direction of right.
* **Velocity vector**: An object's velocity is a measure of its speed and direction of movement. The velocity vector would represent the rate of change of 

## 3) Limit output

In [12]:
prompt_short = "Explain vector in max 50 words."

result_short = call_llm(prompt_short)

print("Output:", result_short["output"])
print("Completion tokens:", result_short["completion_tokens"])

# Word count check
words = len(result_short["output"].split())
print(f"Actual words: {words}")
print("Latency:",          result["latency"], "s")

Output: A vector is a mathematical object with magnitude (length) and direction, represented by an arrow in space, used to describe quantities like force, velocity, and acceleration.
Completion tokens: 34
Actual words: 26
Latency: 4.41 s


 ## 3) JSON OUTPUT

In [20]:
import json

def call_llm_json(prompt, model="llama-3.3-70b-versatile"):
    start = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}  # Groq supports this
    )
    latency = time.time() - start
    content = response.choices[0].message.content
    usage   = response.usage
    return {
        "output":             json.loads(content),
        "latency":            round(latency, 2),
        "total_tokens":      usage.total_tokens,
    }

prompt_json = """
Explain vector. Return JSON only:
{"definition": "", "example": "", "use_case": ""}
"""

result_json = call_llm_json(prompt_json)
print(result_json["output"])
print("Total tokens:", result_json["total_tokens"])
print("Latency:",          result["latency"], "s")

{'definition': 'A vector is a mathematical object that has both magnitude and direction, often represented graphically as an arrow in a coordinate system', 'example': '[3, 4] which represents a point 3 units to the right and 4 units up from the origin', 'use_case': 'Physics and engineering for describing forces, velocities, and accelerations, as well as computer graphics for 2D and 3D modeling'}
Total tokens: 167
Latency: 4.41 s


## Embeddings + Rag

In [19]:
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached torch-2.11.0-cp312-cp312-win_amd64.whl.metadata (29 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached sentence_transformers-5.4.1-py3-none-any.whl (571 kB)
Using cached huggingface_hub-1.11.0-py3-none-any.whl (645 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl (8.0 MB)
Using cached torch-2.11.0-cp312-cp312-win_amd64.whl (114.6 MB)
Using cached transformers-5.5.4-py3-none-any.whl (10.2 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached typer-0.24.1-py3-none-any

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\prajyot\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python312\\site-packages\\torch\\jit\\__init__.py'
Check the permissions.


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\prajyot\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

model_emb = SentenceTransformer("all-MiniLM-L6-v2")

# Knowledge base
documents = [
    "Vector is a quantity with magnitude and direction",
    "Vectors are used in physics and engineering",
    "Examples of vectors include velocity and force",
]

doc_embeddings = model_emb.encode(documents)

def retrieve(query):
    q_emb = model_emb.encode([query])[0]

    scores = [
        np.dot(q_emb, d) / (np.linalg.norm(q_emb) * np.linalg.norm(d))
        for d in doc_embeddings
    ]

    return documents[np.argmax(scores)]

# ---- TEST ----
query = "What is vector?"
context = retrieve(query)

print("Retrieved context:", context)

C:\Users\prajyot\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1701.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieved context: Vector is a quantity with magnitude and direction


# Caching

In [12]:
# simple cache
cache = {}

# fake function (pretend this is expensive like LLM)
def get_answer(question):
    print("[API CALL] computing answer...")
    return f"Answer for: {question}"

# cached version
def cached_answer(question):
    if question in cache:
        print("[CACHE HIT]")
        return cache[question]

    result = get_answer(question)
    cache[question] = result
    return result


# ---- TEST ----

# First time → API
a1 = cached_answer("What is a vector?")
print(a1)

print("\n---\n")

# Second time → Cache
a2 = cached_answer("What is a vector?")
print(a2)

[API CALL] computing answer...
Answer for: What is a vector?

---

[CACHE HIT]
Answer for: What is a vector?


# Model Routing

In [24]:
def route(prompt):
    if len(prompt) < 60:
        return "llama-3.1-8b-instant"   # fast + cheap
    elif len(prompt) < 200:
        return "llama-3.1-8b-instant"      # balanced
    else:
        return "llama-3.3-70b-versatile"  # most capable

def smart_llm(prompt):
    model  = route(prompt)
    result = call_llm(prompt, model)
    result["model_used"] = model
    return result

r_simple  = smart_llm("What is 2+2?")
r_complex = smart_llm("Explain gradient descent with maths and code")


# Test both paths
print("Simple  → model:", r_simple["model_used"],
      "| tokens:", r_simple["total_tokens"])
print("Complex → model:", r_complex["model_used"],
      "| tokens:", r_complex["total_tokens"])

Simple  → model: llama-3.1-8b-instant | tokens: 55
Complex → model: llama-3.1-8b-instant | tokens: 804


# Full agent pipeline

In [ ]:
def agent_pipeline(query):
    # 1. Cache check
    if query in cache:
        print("[CACHE] instant response")
        return cache[query]

    # 2. Retrieve context (RAG)
    context = retrieve(query)

    # 3. Build optimized prompt
    prompt = f"""Context: {context}
Question: {query}
Answer in JSON: {{"answer": "", "confidence": ""}}"""

    # 4. Route to right model
    model = route(prompt)

    # 5. Call Groq
    result = call_llm(prompt, model)

    # 6. Cache
    cache[query] = result

    print(f"Model: {model}")
    print(f"Tokens: {result['total_tokens']}")
    print(f"Latency: {result['latency']}s")
    return result

# Run it
agent_pipeline("What is vector?")

Model: llama-3.1-8b-instant
Tokens: 84
Latency: 0.25s


{'output': '{"answer": "A quantity with magnitude and direction", "confidence": "100%"}',
 'latency': 0.25,
 'prompt_tokens': 65,
 'completion_tokens': 19,
 'total_tokens': 84}